In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# Diccionario para mapear las emociones según el número del archivo RAVDESS
emotion_labels = {
    '01': 'Neutralidad',
    '02': 'Calma',
    '03': 'Felicidad',
    '04': 'Tristeza',
    '05': 'Enojo',
    '06': 'Miedo',
    '07': 'Disgusto',
    '08': 'Sorpresa'
}

# Función para extraer las características (MFCC, Chroma, Tonnetz) del archivo de audio
def extract_features(file_name):
    y, sr = librosa.load(file_name, sr=None)
    
    # Extraer características
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    
    # Combinar todas las características
    features = np.hstack([np.mean(mfccs, axis=1), np.mean(chroma, axis=1),
                          np.mean(spec_contrast, axis=1), np.mean(tonnetz, axis=1)])
    
    return features

# Directorio donde se encuentran los archivos de audio RAVDESS organizados por actor
audio_dir = '../../data/audio'

# Listas para guardar las características y etiquetas
features_list = []
labels_list = []

# Recorrer el directorio principal de los actores y procesar cada archivo de audio
for actor_dir in os.listdir(audio_dir):
    actor_path = os.path.join(audio_dir, actor_dir)
    
    if os.path.isdir(actor_path):  # Si es un directorio (Actor_01, Actor_02, etc.)
        for file in os.listdir(actor_path):
            if file.endswith('.wav'):
                file_path = os.path.join(actor_path, file)
                
                # Extraer características del archivo de audio
                features = extract_features(file_path)
                
                # Obtener la emoción a partir del nombre del archivo
                emotion_code = file.split('-')[2]  # El tercer número es la emoción
                emotion_label = emotion_labels.get(emotion_code, 'Desconocida')
                
                # Añadir las características y las etiquetas
                features_list.append(features)
                labels_list.append(emotion_label)

# Convertir las listas a DataFrame
df = pd.DataFrame(features_list)
df['label'] = labels_list

# Convertir las etiquetas a números
df['label'] = df['label'].astype('category').cat.codes

# Separar las características (X) y las etiquetas (y)
X = df.drop(columns=['label']).values
y = df['label'].values

# Paso 1: Separar los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convertir a tensores de PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Paso 2: Definir el modelo LSTM
class EmotionRecognitionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(EmotionRecognitionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Definir la capa LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Definir la capa fully connected (FC)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x.unsqueeze(1), (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

# Parámetros del modelo
input_size = X_train.shape[1]  # Número de características (MFCC + Chroma + Tonnetz)
hidden_size = 256
num_layers = 2
num_classes = len(np.unique(y))  # Número de emociones

# Instanciar el modelo
model = EmotionRecognitionLSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, num_classes=num_classes)

# Paso 3: Configurar el entrenamiento
learning_rate = 0.001
num_epochs = 30
batch_size = 64

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Entrenamiento
for epoch in range(num_epochs):
    model.train()
    
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

# Paso 4: Guardar el modelo entrenado
torch.save(model.state_dict(), 'modelo_emociones_ravdess.pth')

# Evaluación del modelo
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')


KeyboardInterrupt: 